The point of this notebook is to optimize the volumteHT, an adaptation of Python's dictionary class meant for navigating 3D space.
My implementation divides the space into squares, and the objects in the dictionary are sorted by these square boxes: the box is the key
and the list of items in the box is the items.
If the box is empty, it will be saved as a None object.
Can search with coordinates or box number.
I will start out by density (in terms of avg particles/box) being an argument, and then I will experimentally optimize later.
In this version, we already know the concentration to some approximate value.

Right now, I'm not going to implement collision functionality since I want this to be a generic algorithm for the prpose of modularity. I AM going to keep density as a parameter, however, because I think that that's reasonable given that this is meant to simulate particles in volume.

To find the box size $s$ from the concentration $c$ and targetDensity $d$, have

\begin{gather*}
d = \frac{\text{particles}}{\text{number of boxes}} = c\left(\frac{\text{volume}}{\text{number of boxes}}\right) = cs^3  \implies s = (d/c)^{1/3}
\end{gather*}

Since we want an integer number of boxes in each dimension, we use

\begin{gather*}
b_{x} = \text{round}\left(\frac{x}{\sqrt[3]{\frac{d}{c}}}\right)
\end{gather*}

The code assumes that the box is centered at (0, 0, 0) and that (x, y, z) is the total side lengths, so, for example, the c dimension ranges from -x/L to x/L

However, I decided to 0-index the boxes, instead of having them range from negative to positve numbers. This decision was made to maintain congruence between even and odd numbers of boxes (ie for an even number of boxes, a zero wouldn't make sense, while the opposite is true for an odd number of boxes). Thus, the boxes range from $(b_{x}, b_{y}, b_{z}), b_{i} \in \{0, 1, \ldots, s_{i}-1\}$.

#### Building the $\texttt{Box}$ class
The $\texttt{Box}$ class represents one of the boxes within the volume hash table. While the creation of a $\texttt{Box}$ is not strictly necessary for the code to work, it is added for modularity.

##### Object Variables:
The most important object variable in the $\texttt{box}$ class is $\texttt{contents}$. $\texttt{contents}$ contains a $\texttt{list}$ of items, such as particles, for example. For this implementation, we use $\texttt{Sorted Container}$'s $\texttt{SortedList}$ object. This enables insertion and deletion $O(n\log n)$.

Refer to box with its boxCoords: just 3d ennumeration of boxes

(The boxes are also numbered. Again, this is not strictly necessary for functionality but is added for clarity, organization, and debugging. The $\texttt{box}$ will also include its bounds in each dimension. The bounds are not necessary, but will not change throughout the program so will not be significantly slow.)


##### Object Methods:
$\texttt{Box}$ has an $\texttt{add}$ methods which allows an object to be added to the box. It also has a $\texttt{delete}$ method.

$\texttt{box}$ also has a $\texttt{population}$ method that uses $\texttt{SortedLists}$'s $\texttt{len}$ dunder method (I'm pretty sure it runs $O(1)$.)


###  $\texttt{VolumeHT}$ Instance Variables & Methods

#### Instance Variables
 Since $\texttt{VolumeHT}$ is meant for working in 3 dimensions, we must save the three-dimensional spacial information as well as the partitioning information discuessed above.

 ##### $\texttt{conc}$
$\texttt{c}$ saves the expected concentration in terms of
particles/volume. Saved as a $\texttt{float}$.

##### $\texttt{lengths}$
$\texttt{lengths}$ saves the lengths of the box in each dimension. Information is stored as an $(x, y, z) \; \texttt{tuple}$ since the colume is immutable.

##### $\texttt{numberOfBoxesPerDim}$
Stores the number of boxes in each dimension. Saved as a $\texttt{tuple} \; (b_{x}, b_y, b_z)$.

##### $\texttt{sideLengths}$
Also a tuple that saves the side lengths $(s_{x}, s_y, s_z)$

##### $\texttt{boxDict}$
This is the main data structure underlying the volume HT. It uses the
$\texttt{box}$ number as a key and the $\texttt{box}$ itself as a value.
Uses as $\texttt{dict}$, Python's hash table.


#### Object Methodds
##### $\texttt{coordsToBox()()}$
Function that takes in $(x, y, z)$ coordinates and tells you what box the
particle is in. Ennumeration scheme of boxes delineated above. 

##### $\texttt{adjacent()}$
Takes a box and gives a list of the given box.

##### $\texttt{put()}$
Inserts an object with given coordinates into the volume HT

##### $\texttt{delete()}$
Deletes an object with given coordinates from the volume HT

##### $\texttt{contents()}$
Returns a $\texttt{list}$ of all the particle objects in the volume HT



In [1]:
import numpy
from sortedcontainers import SortedList


In [6]:
class Box:
    
    # initialize the box with its boxCoords, an ennumeration of the boxes in the volumeHT
    def __init__(self, boxCoords):
        self.boxCoords = boxCoords
        self.contents = SortedList()
    
    # returns the number of particles in the box
    def population(self):
        return len(self.contents)

    # add an item to the box
    def add(self, item):
        self.contents.add(item)
    
    # delete a specific item from the box
    def delete(self, item):
        self.contents.remove(item)
    
    # evaluates if the box is empty
    def isEmpty(self):
        return len(self.contents) == 0
    
    

New modification: in order to optimize functionality, there will be two $\texttt{Box}$ classes.

For the local region, that beside the electrode, we consider a collision if the particles overlap in space; ie a nonzero radius is used. Because of this, we need to search through every patricle in a given region and check if the distance between their centers is less than the sum of the radii:
\begin{gather*}
|(x_{1}, y_1, z_1) -(x_2, y_2, z_2)| < r_1 + r_{2}
\end{gather*}
For this, we will used a $\texttt{SortedList}$. We will need to iterate through all the particles anyway, so the order that they are in doesn't matter for the $\texttt{add}$ method. However, the sorting functionality will allow quicker retrieval for $\texttt{delete}$ than a regular Python $\texttt{list}$.

For the bulk region, we care less about details. Therefore, we use Python's built-in $\texttt{dict}$ class (symbol table). Since $\texttt{dict}$'s only allows one key-item pair per key, hashing the $\texttt{atom}$ by the its center coordinates will assure that no two particles are $\textit{centered}$ in the same coordinate, but not that they don't overlap. Checking for overlap will take $O(1)$ amortized. Python's $\texttt{dict}$ class will also allow easy removal at $O(1)$ amortized. The only catch is that the $\texttt{dict}$ class does not have in-built $\texttt{size()}$ functionality, but that is easy to integrate as a object variable that is modified as part of $\texttt{add}$ and $\texttt{delete}$.

Since there are two $\texttt{Box}$ classes, $\texttt{boxType}$ will by a variable passed into $\texttt{volumeHT}$.

Wait lol I should just use one dictionary for the who of the bulk region.

In [ ]:
import numpy

class radiusBox:

    # initialize the box with its boxCoords, an ennumeration of
    # the boxes in the volumeHT
    def __init__(self, number):
        self.boxNumber = number
        self.contents = SortedList()

    # returns the number of particles in the box
    def population(self):
        return len(self.contents)

    # add an item to the box
    def add(self, item):
        self.contents.add(item)

    # delete a specific item from the box
    def delete(self, item):
        self.contents.remove(item)

    # evaluates if the box is empty
    def isEmpty(self):
        return len(self.contents) == 0

In [3]:
class overlapBox:
    
    #initialize the box with its boxCoords, an ennumeration of the boxes
    #in volumeHT
    def __init__(self, number):
        self.boxNumber = number
        self.contents = {}
        self.size = 0
    
    #returns the number of particles in the box
    def population(self):
        return self.size
    
    #add an item to the box
    def add(self, item):
        self.contents[item] = item
    
    #delete an item from the box
    def delete(self, item):
        try:
            self.contents.pop(item)
        except:
            return
            
            

In [34]:
import numpy
from sortedcontainers import SortedList

class volumeHT:
    
    def __init__(self, x, y, z, c, d, boxType):
        self.conc = c #concentration
        self.lengths = tuple([x, y, z]) #save all dimensions for iterating
        tempS = numpy.cbrt(d/c) #compute the approximate box size using
        # the above formula (temp here is temporary not temperature)
        s = [] #save the actual box dimensions
        numberOfBoxes = []
        for length in self.lengths: #follow above formula to find
                                    # integer number of boxes in each dimension
            b = numpy.round(length/tempS)
            sDim = length/b #also find real box length
            numberOfBoxes.append(b) #save these values
            s.append(sDim)
        self.numberOfBoxesPerDim = tuple(numberOfBoxes) #save to object variable
        self.sideLengths = tuple(s)
        self.boxDict = {}
        for i in range(self.numberOfBoxesPerDim[0]):
            for j in (self.numberOfBoxesPerDim[1]):
                for k in (self.numberOfBoxesPerDim[2]):
                    self.boxDict[(i, j, k)] = boxType((i, j, k))
        self.allParticles = []
    
    # find the box corresponding to the coordinates
    def coordsToBox(self, coords):
        boxCoords = []
        coords = coords + numpy.array([self.lengths[0]/2,
                                       self.lengths[1]/2, self.lengths[2]])
        for i in range(3):
            boxCoords.append(numpy.floor(coords[i]/self.sideLengths[i]))
        return numpy.array(boxCoords)
    
    #find the adjacent boxes
    def adjacent(self, box):
        lst = []
        for i in [-1, 0, 1]:
            new_x = box[0] + i
            if new_x < 0 or new_x >= self.numberOfBoxesPerDim[0]:
                continue
            for j in [-1, 0, 1]:
                new_y = box[1] + j
                if new_y < 0 or new_y >= self.numberOfBoxesPerDim[1]:
                    continue
                for k in [-1, 0, 1]:
                    new_z = box[2] + k
                    if new_z < 0 or new_z >= self.numberOfBoxesPerDim[2]:
                        continue
                    if (i, j, k) != (0, 0, 0):
                        lst.append((new_x, new_y, new_z))
        return lst

    # put a particle in the hash table
    def put(self, item, coords):
        box = self.coordsToBox(coords)
        self.boxDict[box].add(item)
    
    # deletes an item from the hash table
    def delete(self, item, coords):
        box = self.coordsToBox(coords)
        self.boxDict[box].delete(item)
        

    
    
    
    
            
        
        